In [1]:
import os
os.chdir( os.path.join( os.environ["VSC_DATA"], "physioex" ) )

import sys
sys.path.append( os.path.join( os.environ["VSC_DATA"], "physioex", "articles", "protosleepnet", "scripts", "src" ) )

from staging_util import GroupDataset

from physioex.data import PhysioExDataModule
from physioex.train.utils import finetune, test
from physioex.train.models import load_model

import os 

fold = 0
num_folds = 10
group = "alzheimers"

batch_size = 64
num_nodes = 1
max_epoch = 1


gd = GroupDataset(
    datasets = [ group ],
    data_folder = os.path.join( os.environ["VSC_SCRATCH_PROJECTS_BASE"], "2024_111", "guido" ),
    preprocessing = "xsleepnet",
    selected_channels = ["EEG", "EOG", "EMG"],
    sequence_length = 21,
)

gd.set_num_folds( num_folds )

dm = PhysioExDataModule(
    datasets = gd,
    batch_size = batch_size,
    folds = 1,
    num_workers = 1,
)

datamodule_kwargs = {}
datamodule_kwargs["batch_size"] = batch_size
datamodule_kwargs["folds"] = fold
datamodule_kwargs["num_nodes"] = num_nodes

model_kwargs = {
    "in_channels": 3,
    "sequence_length": 21,
    "N" : 1,
    "S" : 2,
    "n_prototypes" : 50,
}

model = load_model(
    model = "physioex.train.networks.prototypev1:ProtoSleepNetV1",
    model_kwargs = model_kwargs,
    ckpt_path = f"articles/protosleepnet/models/fast/protosleepnetv1/shhs/EEG-EOG-EMG/model.ckpt",
    softmax = False,
    summary = False,
)

2025-06-18 13:52:34.990 | WARNING  | physioex.data.datareader:__init__:68 - Sequence length 86400 is greater than the max number of windows 1215 for dataset alzheimers/HOA.
Loading healthy dataset: 100%|██████████| 32/32 [00:14<00:00,  2.22it/s]
2025-06-18 13:52:49.447 | WARNING  | physioex.data.datareader:__init__:68 - Sequence length 86400 is greater than the max number of windows 1356 for dataset alzheimers/AD.
Loading unhealthy dataset: 100%|██████████| 36/36 [00:16<00:00,  2.22it/s]
2025-06-18 13:53:05.649 | WARNING  | physioex.data.datamodule:__init__:54 - The usage of PhysioExDataset as datasets is deprecated. Please use a list of dataset names instead.
Splitting test set into sequences: 100%|██████████| 7/7 [00:00<00:00, 189.43it/s]


Dataset split into 50164 train, 13 valid, and 7 test sequences.


In [2]:
train_kwargs = {
    "datasets" : dm,
    "datamodule_kwargs" : datamodule_kwargs,
    "batch_size" : batch_size,
    "fold" : fold,
    "num_validations" : 2,
    "max_epochs": max_epoch,
    "num_nodes": 1,
    "checkpoint_path": f"articles/protosleepnet/models/group/{group}/staging/fold={fold}/"
}

best_checkpoint = finetune(
    model = model,
    learning_rate= 1e-6,  # if None not updated
    train_kwargs = train_kwargs,
)

model = load_model(
    model = "physioex.train.networks.prototypev1:ProtoSleepNetV1",
    model_kwargs = model_kwargs,
    ckpt_path = best_checkpoint,
    softmax = False,
    summary = False,
)

test(
    datasets = dm,
    datamodule_kwargs = datamodule_kwargs,
    model = model,  # if passed model_class, model_config and resume are ignored
    batch_size = 1,
    fold = fold,
    checkpoint_path = f"articles/protosleepnet/models/group/{group}/staging/fold={fold}/test_logs/",
    results_path = f"articles/protosleepnet/models/group/{group}/staging/fold={fold}/test_logs/",
)


[rank: 0] Seed set to 42


2025-06-18 13:53:26.263 | INFO     | physioex.train.utils.train:train:125 - Available devices: [0]


The history saving thread hit an unexpected error (OperationalError('disk I/O error')).History will not be written to the database.


GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
/data/leuven/365/vsc36564/miniconda3/envs/physioex/lib/python3.12/site-packages/pytorch_lightning/callbacks/model_checkpoint.py:654: Checkpoint directory /vscmnt/leuven_icts/_data_leuven/365/vsc36564/physioex/articles/protosleepnet/models/group/alzheimers/staging/fold=0 exists and is not empty.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]

  | Name | Type                      | Params | Mode 
-----------------------------------------------------------
0 | nn   | NNv1                      | 4.0 M  | train
1 | wacc | MulticlassAccuracy        | 0      | train
2 | macc | MulticlassAccuracy        | 0      | train
3 | wf1  | MulticlassF1Score         | 0      | train
4 | mf1  | MulticlassF1Score         | 0      | train
5 | ck   | MulticlassCohenKappa      | 0      | train
6 | pr   | MulticlassPrecision       | 0      | train
7 | rc   | MulticlassRecall          | 0      | train

Epoch 0: 100%|██████████| 784/784 [12:42<00:00,  1.03it/s, v_num=0_1, train_cov_m=0.475, train_cov_v=0.0905, train_loss=1.170, train_acc=0.556, train_cl=0.00748, val_cov_m=0.475, val_cov_v=0.00482, val_loss=3.030, val_acc=0.645, val_cl=0.0191]

`Trainer.fit` stopped: `max_epochs=1` reached.


Epoch 0: 100%|██████████| 784/784 [12:42<00:00,  1.03it/s, v_num=0_1, train_cov_m=0.475, train_cov_v=0.0905, train_loss=1.170, train_acc=0.556, train_cl=0.00748, val_cov_m=0.475, val_cov_v=0.00482, val_loss=3.030, val_acc=0.645, val_cl=0.0191]


[rank: 0] Seed set to 42
2025-06-18 14:07:03.926 | INFO     | physioex.train.utils.test:test:97 - Available devices: [0]
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
2025-06-18 14:07:04.862 | INFO     | physioex.train.utils.test:test:119 - Testing on (tensor([[[[ 2.7510e-02,  1.2610e-01,  2.7029e-01,  ...,  4.1806e-02,
           -1.0741e-01,  1.0673e+00],
          [-5.1742e-02,  6.5522e-01,  6.6868e-01,  ...,  6.1584e-01,
            3.9173e-01,  8.4321e-01],
          [ 5.8034e-01,  7.0756e-01,  7.0797e-01,  ...,  2.5893e-01,
            6.1777e-01,  6.3326e-01],
          ...,
          [ 1.4287e-01, -1.6928e-01, -9.1471e-02,  ...,  2.9362e-01,
            1.3049e-01,  1.2490e-01],
          [-1.6598e-01, -5.1403e-02,  1.1226e-01,  ...,  2.8778e-02,
            2.3857e-01,  5.3179e-01],
          [-2.2351e-01, -2.7341e-01, -1.2321e-01,  ..., -1.8757e-01,
           -2.3548e-02,  1.8555e-01]],

         [[ 6.1644

Testing DataLoader 0: 100%|██████████| 7/7 [02:00<00:00,  0.06it/s]
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
       Test metric             DataLoader 0
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
        test_acc            0.6966977715492249
         test_ck             0.565288245677948
         test_cl            0.02061326801776886
       test_cov_m           0.4743000566959381
       test_cov_v           0.00495593948289752
         test_f1            0.6702818870544434
        test_loss            2.895670175552368
        test_macc           0.5534483790397644
        test_mf1            0.5475702285766602
         test_pr            0.6772080659866333
         test_rc            0.6966977715492249
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────


,test_cov_m,test_cov_v,test_loss,test_acc,test_cl,test_f1,test_ck,test_pr,test_rc,test_macc,test_mf1,dataset,fold
0,0.4743,0.004956,2.89567,0.696698,0.020613,0.670282,0.565288,0.677208,0.696698,0.553448,0.54757,"([[tensor([[ 0.0275, 0.1261, 0.2703, ..., ...",0
